In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        # print(os.path.join(dirname, filename))
        pass

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
%%capture
!pip install unsloth datasets trl -q

In [3]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import gc
import torch

torch.cuda.empty_cache()
gc.collect()

from unsloth import FastLanguageModel, is_bfloat16_supported
from datasets import load_dataset, concatenate_datasets
from trl import SFTTrainer
from transformers import TrainingArguments

# =========================
# MODEL LOAD
# =========================

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-4-E2B-it",
    max_seq_length=128,
    dtype=None,
    load_in_4bit=True,
    device_map={"": 0},   # IMPORTANT FIX
)

model = FastLanguageModel.get_peft_model(
    model,
    r=4,                         # reduced from 8
    target_modules=["q_proj", "v_proj"],
    lora_alpha=8,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print(f"Allocated VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")
model.print_trainable_parameters()

# =========================
# DATASET
# =========================

truthfulqa = load_dataset(
    "truthful_qa",
    "generation",
    split="validation"
)

def format_truthfulqa(x):
    return {
        "text": (
            f"<start_of_turn>user\n"
            f"{x['question']}<end_of_turn>\n"
            f"<start_of_turn>model\n"
            f"{x['best_answer']}\n\n"
            f"REASONING:\n"
            f"- Based on: established facts\n"
            f"- Uncertain about: individual variation\n"
            f"- User should: consult a professional"
            f"<end_of_turn>"
        )
    }

truthfulqa_fmt = truthfulqa.map(
    format_truthfulqa,
    remove_columns=truthfulqa.column_names
)

# Reduced dataset size for Kaggle stability
medqa = load_dataset(
    "lavita/ChatDoctor-HealthCareMagic-100k",
    split="train"
).select(range(300))

def format_medqa(x):
    return {
        "text": (
            f"<start_of_turn>user\n"
            f"Medical question: {x['input']}<end_of_turn>\n"
            f"<start_of_turn>model\n"
            f"{x['output'][:200]}\n\n"
            f"REASONING:\n"
            f"- Based on: clinical knowledge\n"
            f"- Uncertain about: individual factors\n"
            f"- User should: consult a physician"
            f"<end_of_turn>"
        )
    }

medqa_fmt = medqa.map(
    format_medqa,
    remove_columns=medqa.column_names
)

combined = concatenate_datasets([
    truthfulqa_fmt,
    medqa_fmt
]).shuffle(seed=42)

print(f"Combined dataset size: {len(combined)}")

torch.cuda.empty_cache()
gc.collect()

# =========================
# TRAINER
# =========================

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=combined,
    dataset_text_field="text",
    max_seq_length=128,
    dataset_num_proc=1,      # reduced workers
    packing=False,           # IMPORTANT: reduces VRAM spikes
    args=TrainingArguments(
        num_train_epochs=3,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,  # reduced from 8
        warmup_steps=5,
        learning_rate=2e-4,

        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),

        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",

        seed=42,
        output_dir="/kaggle/working/outputs",
        report_to="none",

        remove_unused_columns=False,
        dataloader_pin_memory=False,
        max_grad_norm=0.3,
    ),
)

# =========================
# TRAIN
# =========================

print("Starting training...")
trainer_stats = trainer.train()

print(
    f"Done! "
    f"Time: {trainer_stats.metrics['train_runtime']:.1f}s "
    f"Loss: {trainer_stats.metrics['train_loss']:.4f}"
)

torch.cuda.empty_cache()
gc.collect()



🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/203 [00:00<?, ?B/s]

...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,117 | Num Epochs = 3 | Total steps = 420
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 1,210,368 of 5,124,388,384 (0.02% trained)


Step,Training Loss
10,1.934770
20,1.557708
30,1.271667
40,0.988332
50,0.805424
60,0.690100
70,0.526211
80,0.475779
90,0.498899
100,0.380342


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/outputs/checkpoint-420/tokenizer_config.json.


Done! Time: 688.4s Loss: 0.5089


109

In [4]:
# =========================
# FREE MEMORY
# =========================


torch.cuda.empty_cache()
gc.collect()

# =========================
# SAVE MERGED MODEL
# =========================

print("Saving merged 16-bit model...")

model.save_pretrained_merged(
    "/kaggle/working/truthlayer-med-f16",
    tokenizer,
    save_method="merged_16bit",
)

print("Merged model saved!")

Saving merged 16-bit model...


config.json: 0.00B [00:00, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/truthlayer-med-f16/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Splitting model.safetensors (size: 9.54 GB)...


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [01:20<00:00, 80.26s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 5/5 [01:41<00:00, 20.36s/it]


Unsloth: Regenerating safetensors index...
Unsloth: Merge process complete. Saved to `/kaggle/working/truthlayer-med-f16`
Merged model saved!
